# General tetrahedral degree and admissible P5/P2 estimation

P5/P2 satisfies the sufficient three-dimensional condition $k\geq\ell+3$. This notebook validates archived results and displays broken physical fields; it does not rerun the campaign.

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
from IPython.display import Image, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
DATA = ROOT / "examples/results/tetra-pk"
uniform = json.loads((DATA / "uniform.json").read_text())
fixed = json.loads((DATA / "fixed.json").read_text())
recovered = json.loads((DATA / "reconstruction-order.json").read_text())
assert len(uniform["rows"]) == 5 and len(fixed["rows"]) == 2
rows = uniform["rows"] + fixed["rows"] + [recovered]
for row in rows:
    assert row["local_degree"] >= row["trace_degree"] + 3
    path = DATA / row["archive"]
    assert hashlib.sha256(path.read_bytes()).hexdigest() == row["archive_sha256"]
    with np.load(path) as archive:
        digest = hashlib.sha256(np.ascontiguousarray(archive["rt_basis"]).tobytes()).hexdigest()
        assert digest == row["basis_sha256"]
    assert row["continuous_equilibrium"] < 1e-9
    assert row["normal_moments"] < 1e-9
[(r["name"], 100*r["pressure_relative"], 100*r["raw_flux_relative"], 100*r["rt_flux_relative"], r["effectivity"]) for r in rows]

RT3 reuses the exact same P5 pressure and skeletal trace. Canonical reconstruction imposes physical moments and is not the unconstrained L2 projection; continuous-test conservation is distinct from cellwise DG0 balance.

In [ ]:
assert recovered["new_primal_solve"] is False
with np.load(DATA / recovered["parent_archive"]) as a, np.load(DATA / recovered["archive"]) as b:
    for name in a.files:
        if name.startswith("pressure_") or name == "trace":
            assert np.array_equal(a[name], b[name])
for name in ("uniform-convergence", "fixed-resolution", "fixed-components-rt2", "fixed-components-rt3", "fixed-profiles"):
    display(Image(filename=ROOT / "docs/figures/tetra-pk" / f"{name}.png"))